In [2]:
with open('tiny_shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()


In [3]:

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:

# tokenisation
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)


torch.Size([1115394]) torch.int64


In [6]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8 # context length
train_data[:block_size+1] # 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")
    # that one example is practically 8 examples

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [9]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # random numbers between 0 and len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [10]:
# bigram again 
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) # batch ,time , channels, (4,8,vocab_size)
       
        if targets is None:
            loss = None
        else:
            B,T,C= logits.shape
            logits= logits.view(B*T, C)
            targets = targets.view(B*T) # can also write -1

            loss= F.cross_entropy(logits, targets) # expects (B,C,T)

        return logits , loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    

m = BigramLanguageModel(vocab_size)
logits, loss =m(xb,yb) # we need this 
print(logits.shape)
print(loss)
print(-math.log(((1/65)))) # expected loss
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())) # not useful because its totally random till now

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
4.174387269895637

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [11]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) # optimiser earlier used gradient descent 

In [12]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.382369041442871


In [13]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist())) # something not random at least 


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


#### Self Attention


In [14]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time ,channels
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 32])

In [15]:
# x[b,t] =mean{i<=t}.x[b,i]
#averaging past context with for loops
# bow -> bag of words
xbow=torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev= x[b,:t+1] # shape is (t,C)
        xbow[b,t]=torch.mean(xprev,0)

In [16]:
# making it efficient using matrix multiplication
torch.manual_seed(42)
a= torch.ones(3,3)
a= torch.tril(a)
a=a/torch.sum(a,1,keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print(a)
print(b)
print(c)

# c is just column wise sum in each row because a is all ones
# but on making a lower triangular we are get sum of variable number of rows and we can do avg if we normalise the rows

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [17]:
wei= torch.tril(torch.ones(T,T))
wei=wei/wei.sum(1, keepdim=True)
wei # all rows sum to one equivalent to a above
xbow2=wei @ x # (B(created here)T,T) @ (B,T,C) -> (B,T,C)


In [18]:
# xbow[0] , xbow2[0]

In [19]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T)) # lower triangular ones
wei = torch.zeros((T,T)) # intialising affinity?
wei = wei.masked_fill(tril == 0, float('-inf')) # where tril =0 make them - inf 
wei = F.softmax(wei, dim=-1) # softmax of -inf is 0
xbow3 = wei @ x
print(wei)
print(xbow3)


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[[ 1.8077e-01, -6.9988e-02, -3.5962e-01,  ..., -8.0164e-01,
           1.5236e+00,  2.5086e+00],
         [-2.4116e-01, -1.6063e-01,  3.2526e-01,  ...,  3.6581e-01,
           1.5667e+00,  1.0527e+00],
         [-4.3893e-01,  9.2179e-02,  1.9971e-01,  ...,  9.8210e-02,
           7.1071e-01,  5.6531e-01],
         ...,
         [-9.8921e-01,  1.3417e-01,  2.8014e-01,  ...,  3.4951e-01,
          

In [20]:
# we dont want all uniform because some tokens will find others more or less interesting and that is data dependent 
# so gather information from the past in a data dependent way 
# every single token at each position will emit 2 vectors a query and a key
# query is "what am i looking for?" and key is "what do i contain?" 
# dot product of query and keys of all other tokens is what gives us our affinities so that now becomes wei


In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time ,channels
x=torch.randn(B,T,C)
# when forward this linear on top of x all the tokens in all the places in B,T arrangement in parralel and independently create key and query
# single head of self attention
head_size = 16 # hyper parameter
key = nn.Linear(C,head_size, bias= False) 
query=nn.Linear(C,head_size, bias= False)
value=nn.Linear(C,head_size, bias= False)
k= key(x) # (B,T,16)
q= query(x) # (B,T,16)
wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) -> (B,T,T)
# for every row of B we have T^2 matrix giving us the affinities 

# see wei at every step before masking after it before softmax and after it 
tril = torch.tril(torch.ones(T, T)) # lower triangular ones
# wei = torch.zeros((T,T)) # intialising affinity?
wei = wei.masked_fill(tril == 0, float('-inf')) # where tril =0 make them - inf 
wei = F.softmax(wei, dim=-1) # softmax of -inf is 0

v= value(x) # gets aggregated for the purposes of this single head
# out = wei @ x
out = wei @ v

out.shape

torch.Size([4, 8, 16])

In [32]:
wei[0] # before wei was just a constant but now every single batch element has different wei because they contain different tokens at different positions

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [ ]:
# notes
# attention is a communication mechanism
# attention doesnt have a notion of space it acts as a set of vectors so we have to encode them positionally
# each example across the batch is processed completely independently
# in some cases we might want all the tokens talk to each other fully without constraints there we have to use encoder block that is delete the line wei = wei.masked_fill(tril == 0, float('-inf'))
#       what is implemented here is sometimes called decoder block
# self attention is when keys queries and values all come from the same source
# scaled dot product attention (division by sqrt of head_size) 
#       unit gaussian inputs give var of order of head size, its important because we later pass it to softmax and we want wei to be fairly diffused before that we dont want the values to be too extreme otherwise softmax will be peaky
